# Portfolio Tracker — Notebook Interactivo

Réplica completa de la aplicación web usando `PortfolioClient` (todos los métodos devuelven DataFrames).

**Secciones:**
1. Setup
2. Carga de datos (Órdenes 1238478.tsv)
3. Posiciones del portfolio
4. Resumen general (Asset Allocation)
5. Detalles + Comparación MSCI World
6. Evolución temporal
7. Correlaciones
8. Simulador de incorporación
9. Retirada de fondos (TaxOptimizer)
10. Performance del portfolio
11. Diagnóstico de datos

## 1. Setup

In [4]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

# Add backend to path
BACKEND_DIR = Path(".").resolve().parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from app.client import PortfolioClient

print(f"Backend dir: {BACKEND_DIR}")

Backend dir: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend


## 2. Carga de Datos

Cargamos el fichero de órdenes TSV y creamos el `PortfolioClient` que centraliza
todos los accesos a datos (precios, sectores, regiones, métricas, etc.).

In [5]:
# Cargar órdenes desde el TSV
ORDERS_FILE = BACKEND_DIR / "data" / "Órdenes 1238478.tsv"
CACHE_PATH = str(BACKEND_DIR / "data" / "cache")

client = PortfolioClient(source=str(ORDERS_FILE), cache_path=CACHE_PATH)

print(f"Archivo:      {ORDERS_FILE.name}")
print(f"Posiciones:   {len(client.portfolio.positions)} fondos")
print(f"Lotes FIFO:   {len(client.portfolio.open_lots)}")
print(f"Capital inv.: {client.portfolio.get_total_invested():,.2f} €")

print("\nMovimientos cargados:")
display(client.movements().head(10))

Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0146309002 (light)
Archivo:      Órdenes 1238478.tsv
Posiciones:   18 fondos
Lotes FIFO:   59
Capital inv.: 133,348.66 €

Movimientos cargados:


,Fecha,ISIN,Importe,Participaciones,Estado
0,2024-10-27,FR0000989626,"2,500.00",59.00,Finalizada
1,2024-12-02,IE00BYX5MX67,"2,000.00",142.71,Finalizada
2,2024-12-14,IE00BYX5MX67,500.00,35.48,Finalizada
3,2024-12-26,IE00BYX5MX67,"1,000.00",70.73,Finalizada
4,2025-01-13,IE00BYX5MX67,"1,000.00",71.64,Finalizada
5,2025-02-22,IE00BYX5MX67,"2,000.00",143.36,Finalizada
6,2025-04-01,IE00BYX5MX67,"1,000.00",78.56,Finalizada
7,2025-04-10,IE00BYX5MX67,"1,500.00",130.28,Finalizada
8,2025-05-26,IE00BYX5MX67,"1,500.00",117.45,Finalizada
9,2025-06-25,IE00BYX5MX67,"1,500.00",116.83,Finalizada


## 3. Posiciones del Portfolio

Posiciones actuales con P&L calculado a partir de precios live.

In [6]:
# Posiciones actuales con P&L
df_positions = client.positions(live=True)
df_positions

Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache

--- Procesando: ES0141116030 (light)---
✗ Error obteniendo datos para ES0141116030: Invalid ISIN number: ES0141116030
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache

--- Procesando: ES0146309002 (light)---
✓ Datos obtenidos para ES0146309002: 1971 días
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0146309002 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0146309002 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache

--- Procesando: ES01407

,ISIN,Fondo,Valor_Actual,Capital_Invertido,Ganancia_Euros,Ganancia_Pct,Participaciones,Precio_Actual,Fecha_NAV
0,IE00BYX5MX67,Fidelity S&P 500 Index Fund,"28,005.23","25,002.88","3,002.35",12.01,"1,849.30",15.14,2026-05-01
1,IE00BD0NCM55,iShares Developed World Index Fund (IE),"18,218.88","16,402.45","1,816.43",11.07,689.90,26.41,2026-04-30
2,IE00BYX5NX33,Fidelity MSCI World Index Fund,"17,469.22","16,141.22","1,328.00",8.23,"1,332.13",13.11,2026-05-01
3,LU0302296495,DNB Fund - Technology,"12,120.31","11,000.00","1,120.31",10.18,7.14,"1,698.47",2026-05-04
4,ES0146309002,Horos Value Internacional FI,"11,795.25","10,820.38",974.87,9.01,54.52,216.33,2026-04-30
5,LU0273159177,DWS Invest - Gold and Precious Metals Equities,"6,172.88","7,019.02",-846.15,-12.06,20.93,295.00,2026-05-04
6,IE00BH6XSF26,Heptagon Fund ICAV - Kopernik Global All-Cap E...,"6,682.66","6,509.51",173.15,2.66,21.41,312.11,2026-04-30
7,LU1598719752,Cobas LUX SICAV - Cobas International Fund,"5,805.16","5,500.00",305.16,5.55,32.88,176.54,2026-04-30
8,LU2466448532,Echiquier Space,"5,641.59","5,000.00",641.59,12.83,28.18,200.17,2026-04-30
9,LU0840158819,Storm Fund II - Storm Bond Fund,"5,070.57","5,000.00",70.57,1.41,32.27,157.11,2026-05-04


In [4]:
df_positions.sum()

ISIN                 IE00BYX5MX67IE00BD0NCM55IE00BYX5NX33LU03022964...
Fondo                Fidelity S&P 500 Index FundiShares Developed W...
Valor_Actual                                                143,103.06
Capital_Invertido                                           132,895.46
Ganancia_Euros                                               10,207.59
Ganancia_Pct                                                    100.81
Participaciones                                               6,663.56
Precio_Actual                                                 3,930.09
Fecha_NAV            2026-05-012026-04-302026-05-012026-05-042026-0...
dtype: object

## 4. Resumen General

In [5]:
# Asset allocation — distribución por tipo de activo
df_alloc = client.asset_allocation()
df_alloc_plot = df_alloc[df_alloc["Valor"] > 0]

fig = go.Figure(data=[go.Pie(
    labels=df_alloc_plot["Tipo"],
    values=df_alloc_plot["Valor"],
    hole=0.45,
    textinfo="label+percent",
    marker=dict(colors=px.colors.qualitative.Set2),
)])
fig.update_layout(
    title="Asset Allocation",
    template="plotly_dark",
    height=400,
)
fig.show()

display(df_alloc)

,Tipo,Valor,Peso_Pct
0,Renta Variable,"115,485.97",80.70
1,Alternativo,"17,968.13",12.56
2,Renta Fija,"9,648.96",6.74


In [7]:
# Peso de cada fondo — gráfico horizontal
total_val = df_positions["Valor_Actual"].sum()
df_positions["Peso_Pct"] = (df_positions["Valor_Actual"] / total_val * 100).round(2)
df_sorted = df_positions.sort_values("Peso_Pct", ascending=True)

fig = go.Figure(go.Bar(
    x=df_sorted["Peso_Pct"],
    y=df_sorted["Fondo"],
    orientation="h",
    text=df_sorted["Peso_Pct"].apply(lambda v: f"{v:.1f}%"),
    textposition="outside",
    marker_color=px.colors.qualitative.Vivid[:len(df_sorted)],
))
fig.update_layout(
    title="Peso de cada fondo en la cartera",
    template="plotly_dark",
    xaxis_title="Peso (%)",
    height=max(350, 40 * len(df_sorted)),
    margin=dict(l=250),
)
fig.show()

## 5. Detalles + Comparación MSCI World

In [7]:
# Comparación con MSCI World (IE00B4L5Y983)
benchmark = client.benchmark_comparison()
df_sectors = benchmark["sectors"]
df_regions = benchmark["regions"]

print(f"Sectores comparados: {len(df_sectors)}")
print(f"Regiones comparadas: {len(df_regions)}")

Sectores comparados: 11
Regiones comparadas: 13


In [8]:
# Sectores — Mi Cartera vs MSCI World
df_s = df_sectors.sort_values("Mi_Cartera", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Mi Cartera",
    y=df_s["Nombre"],
    x=df_s["Mi_Cartera"],
    orientation="h",
    marker_color="#00d4aa",
))
fig.add_trace(go.Bar(
    name="MSCI World",
    y=df_s["Nombre"],
    x=df_s["Benchmark"],
    orientation="h",
    marker_color="#8b5cf6",
))
fig.update_layout(
    title="Sectores — Mi Cartera vs MSCI World",
    barmode="group",
    template="plotly_dark",
    xaxis_title="Peso (%)",
    height=max(400, 35 * len(df_s)),
    margin=dict(l=200),
    legend=dict(orientation="h", y=1.05),
)
fig.show()

In [9]:
# Regiones — Mi Cartera vs MSCI World
df_r = df_regions.sort_values("Mi_Cartera", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Mi Cartera",
    y=df_r["Nombre"],
    x=df_r["Mi_Cartera"],
    orientation="h",
    marker_color="#00d4aa",
))
fig.add_trace(go.Bar(
    name="MSCI World",
    y=df_r["Nombre"],
    x=df_r["Benchmark"],
    orientation="h",
    marker_color="#8b5cf6",
))
fig.update_layout(
    title="Regiones — Mi Cartera vs MSCI World",
    barmode="group",
    template="plotly_dark",
    xaxis_title="Peso (%)",
    height=max(400, 35 * len(df_r)),
    margin=dict(l=200),
    legend=dict(orientation="h", y=1.05),
)
fig.show()

In [10]:
# Métricas por fondo (estrellas, sharpe, volatilidad, TER...)
df_fund_metrics = client.fund_metrics()
df_fund_metrics

,ISIN,Fondo,Estrellas_MS,Rating_Riesgo,Sharpe,Volatilidad,Rent_1Y,Rent_3Y,TER
0,IE00BYX5MX67,Fidelity S&P 500 Index Fund,4.00,6.00,0.74,14.39,None,None,0.06
1,IE00BD0NCM55,iShares Developed World Index Fund (IE),4.00,5.00,0.72,12.92,None,None,0.12
2,IE00BYX5NX33,Fidelity MSCI World Index Fund,4.00,5.00,0.72,12.98,None,None,0.12
3,LU0302296495,DNB Fund - Technology,4.00,4.00,1.34,15.05,None,None,0.77
4,ES0146309002,Horos Value Internacional FI,5.00,4.00,1.00,11.82,None,None,1.88
5,LU0273159177,DWS Invest - Gold and Precious Metals Equities,4.00,7.00,0.06,31.39,None,None,1.57
6,IE00BH6XSF26,Heptagon Fund ICAV - Kopernik Global All-Cap E...,5.00,4.00,1.17,11.76,None,None,1.66
7,LU1598719752,Cobas LUX SICAV - Cobas International Fund,5.00,4.00,1.03,16.68,None,None,1.64
8,LU2466448532,Echiquier Space,5.00,4.00,1.65,20.95,None,None,1.73
9,LU0840158819,Storm Fund II - Storm Bond Fund,NaN,2.00,2.30,7.02,None,None,0.57


## 6. Evolución Temporal

In [18]:
df_hist[['Fidelity S&P 500 Index Fund']].drop_duplicates()

,Fidelity S&P 500 Index Fund
0,NaN
1048,12.77


In [8]:
# Histórico de precios normalizado (base 100)
COLORS = [
    "#00d4aa", "#8b5cf6", "#ff6b6b", "#ffd93d", "#6bcb77",
    "#4ecdc4", "#ff8a5c", "#a8e6cf", "#fdcb6e", "#e17055",
    "#0984e3", "#6c5ce7", "#fd79a8", "#00cec9", "#fab1a0",
]

df_hist = client.history(years=5)

if not df_hist.empty:
    date_col = df_hist.columns[0]
    all_price_cols = [c for c in df_hist.columns if c != date_col]

    # Ordenar fondos: "Mi Cartera Actual" primero, luego por peso descendente
    weight_map = dict(zip(df_positions["Fondo"], df_positions.get("Peso_Pct", pd.Series(dtype=float)))) if "Peso_Pct" in df_positions.columns else {}

    def _fund_sort_key(col: str) -> tuple:
        if "cartera" in col.lower():
            return (-9999.0, col)
        return (-weight_map.get(col, 0.0), col)

    price_cols = sorted(all_price_cols, key=_fund_sort_key)

    # Normalizar a base 100
    df_norm = df_hist.copy()
    for col in price_cols:
        valid = df_norm[col].dropna()
        first = valid.iloc[0] if len(valid) > 0 else 1
        if first and first != 0:
            df_norm[col] = (df_norm[col] / first) * 100

    fig = go.Figure()
    for idx, col in enumerate(price_cols):
        fig.add_trace(go.Scatter(
            x=df_norm[date_col],
            y=df_norm[col],
            name=col,
            mode="lines",
            line=dict(color=COLORS[idx % len(COLORS)], width=1.5),
        ))

    fig.update_layout(
        title="Evolución Normalizada (Base 100)",
        template="plotly_dark",
        xaxis_title="Fecha",
        yaxis_title="Valor (base 100)",
        height=550,
        legend=dict(orientation="h", y=-0.15),
        hovermode="x unified",
    )
    fig.show()
else:
    print("No hay datos de historial disponibles.")


Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0140794001 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0141116030 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0146309002 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE000ZYRH0Q7 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE00B88WFS66 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive -

In [9]:
# Evolución en valor absoluto (NAV / €)
if not df_hist.empty:
    fig = go.Figure()
    for idx, col in enumerate(price_cols):  # reutiliza price_cols ordenado por peso
        fig.add_trace(go.Scatter(
            x=df_hist[date_col],
            y=df_hist[col],
            name=col,
            mode="lines",
            line=dict(color=COLORS[idx % len(COLORS)], width=1.5),
        ))

    fig.update_layout(
        title="Evolución en Valor Absoluto (NAV / €)",
        template="plotly_dark",
        xaxis_title="Fecha",
        yaxis_title="NAV / €",
        height=550,
        legend=dict(orientation="h", y=-0.15),
        hovermode="x unified",
    )
    fig.show()
else:
    print("No hay datos de historial disponibles.")


In [ ]:
# ── Métricas de evolución: Rentabilidad, Volatilidad, Sharpe, Alpha, Beta ──
# Delegamos toda la lógica a client.evolution_metrics() — thin wrapper sobre el backend.
import ipywidgets as widgets
from IPython.display import display as ipy_display, clear_output

RISK_FREE_ANNUAL = 0.03   # tipo libre de riesgo anual; configurable aquí

df_evol = client.evolution_metrics(years=5, risk_free_annual=RISK_FREE_ANNUAL)
bm_col  = df_evol.attrs.get("benchmark", "—")

if not df_evol.empty:
    metric_cols = [c for c in df_evol.columns if c not in ("Fondo",)]

    # ── Widgets de control ────────────────────────────────────────────────
    dd_sort = widgets.Dropdown(
        options=metric_cols,
        value="CAGR_Pct",
        description="Ordenar por:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="280px"),
    )
    dd_sort2 = widgets.Dropdown(
        options=["(ninguno)"] + metric_cols,
        value="(ninguno)",
        description="2ª clave:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="240px"),
    )
    toggle_asc = widgets.ToggleButton(
        value=False,
        description="↑ Ascendente",
        layout=widgets.Layout(width="140px"),
    )
    dd_cols = widgets.SelectMultiple(
        options=metric_cols,
        value=list(metric_cols),
        description="Columnas:",
        style={"description_width": "initial"},
        rows=len(metric_cols),
        layout=widgets.Layout(width="240px"),
    )
    out = widgets.Output()

    def _render(_change=None):
        with out:
            clear_output(wait=True)
            keys = [dd_sort.value]
            if dd_sort2.value != "(ninguno)":
                keys.append(dd_sort2.value)
            visible = ["Fondo"] + [c for c in metric_cols if c in list(dd_cols.value)]
            df_show = (
                df_evol[visible]
                .sort_values(keys, ascending=toggle_asc.value)
                .reset_index(drop=True)
            )

            def _color(val):
                if not isinstance(val, (int, float)) or val != val:
                    return ""
                return "color: #00d4aa" if val >= 0 else "color: #ff6b6b"

            ipy_display(
                df_show.style
                .applymap(_color, subset=[c for c in visible if c != "Fondo"])
                .format({c: "{:.2f}" for c in visible if c != "Fondo"}, na_rep="—")
                .set_table_styles([
                    {"selector": "th", "props": [("background-color", "#1e1e2e"), ("color", "#cdd6f4")]},
                    {"selector": "td", "props": [("background-color", "#181825"), ("color", "#cdd6f4")]},
                ])
            )

    for w in [dd_sort, dd_sort2, toggle_asc, dd_cols]:
        w.observe(_render, names="value")

    print(f"📊  Métricas de Evolución  (benchmark para α/β: {bm_col})")
    print(f"    Rf anual: {RISK_FREE_ANNUAL*100:.1f}%  |  Ventana: {df_hist[date_col].iloc[0].date()} → {df_hist[date_col].iloc[-1].date()}\n")
    ipy_display(
        widgets.HBox([dd_sort, dd_sort2, toggle_asc]),
        widgets.HBox([widgets.Label("Columnas visibles:"), dd_cols]),
        out,
    )
    _render()
else:
    print("No hay datos de historial para calcular métricas.")


📊  Métricas de Evolución  (benchmark para α/β: Fidelity S&P 500 Index Fund)
    Tipo libre de riesgo anual: 3.0%  |  Ventana: 2021-05-04 → 2026-05-04



Output()

## 7. Correlaciones

In [13]:
# Matriz de correlación
df_corr = client.correlation(years=5)

if not df_corr.empty:
    labels = list(df_corr.columns)
    matrix_np = df_corr.values.astype(float)

    fig = go.Figure(data=go.Heatmap(
        z=matrix_np,
        x=labels,
        y=labels,
        colorscale=[
            [0.0, "#ff4444"],
            [0.5, "#1a1a2e"],
            [1.0, "#00d4aa"],
        ],
        zmin=-1,
        zmax=1,
        text=[[f"{v:.2f}" if not np.isnan(v) else "N/A" for v in row] for row in matrix_np],
        texttemplate="%{text}",
        hovertemplate="%{x} vs %{y}: %{z:.2f}<extra></extra>",
    ))

    fig.update_layout(
        title="Matriz de Correlación (rendimientos diarios)",
        template="plotly_dark",
        height=max(500, 50 * len(labels)),
        width=max(500, 50 * len(labels)),
        xaxis=dict(tickangle=-45),
    )
    fig.show()
else:
    print("Datos insuficientes para calcular correlaciones.")

Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0140794001 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0141116030 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0146309002 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE000ZYRH0Q7 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE00B88WFS66 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive -

In [14]:
# Pares más y menos correlacionados
if not df_corr.empty:
    labels = list(df_corr.columns)
    pairs = []
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            val = df_corr.iloc[i, j]
            if not np.isnan(val):
                pairs.append((labels[i], labels[j], val))

    pairs.sort(key=lambda x: x[2])

    print("\nPares MENOS correlacionados (mejor diversificación):")
    for a, b, v in pairs[:3]:
        print(f"  {a} <-> {b}: {v:.3f}")

    print("\nPares MÁS correlacionados (riesgo concentración):")
    for a, b, v in pairs[-3:]:
        print(f"  {a} <-> {b}: {v:.3f}")


Pares MENOS correlacionados (mejor diversificación):

Pares MÁS correlacionados (riesgo concentración):


## 8. Simulador de Incorporación

In [15]:
# Simular incorporación de un fondo
SIM_ISIN = "IE00B4L5Y983"  # iShares Core MSCI World
SIM_AMOUNT = 10_000.0

print(f"Simulando incorporación de {SIM_ISIN} con {SIM_AMOUNT:,.0f} €...")
sim = client.simulate_addition(SIM_ISIN, SIM_AMOUNT)

meta = sim["metadata"]
print(f"\nFondo añadido:     {meta['added_name']}")
print(f"Total actual:      {meta['current_total']:,.2f} €")
print(f"Total simulado:    {meta['simulated_total']:,.2f} €")

print("\nMétricas comparadas:")
display(sim["metrics"])

Simulando incorporación de IE00B4L5Y983 con 10,000 €...
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0146309002 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache

--- Procesando: ES0141116030 (light)---
✗ Error obteniendo datos para ES0141116030: Invalid ISIN number: ES0141116030
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache

--- Procesando: ES0146309002 (light)---
✓ Datos obtenidos para ES0146309002: 1971 días
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0146309002 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Document

AttributeError: 'NoneType' object has no attribute 'get'

In [ ]:
# Pesos: Actual vs Simulado
df_sim = sim["weights"].sort_values("Peso_Simulado", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Actual",
    y=df_sim["Fondo"],
    x=df_sim["Peso_Actual"],
    orientation="h",
    marker_color="#00d4aa",
))
fig.add_trace(go.Bar(
    name="Simulado",
    y=df_sim["Fondo"],
    x=df_sim["Peso_Simulado"],
    orientation="h",
    marker_color="#8b5cf6",
))
fig.update_layout(
    title=f"Pesos: Actual vs Simulado (+{SIM_AMOUNT:,.0f} € en {meta['added_name']})",
    barmode="group",
    template="plotly_dark",
    xaxis_title="Peso (%)",
    height=max(400, 35 * len(df_sim)),
    margin=dict(l=250),
)
fig.show()

## 9. Retirada de Fondos (TaxOptimizer)

In [ ]:
# Retirada optimizada de fondos
WITHDRAWAL_AMOUNT = 50_000.0

df_tax = client.tax_optimize(WITHDRAWAL_AMOUNT)

print(f"{'=' * 60}")
print(f"  RETIRADA OPTIMIZADA: {WITHDRAWAL_AMOUNT:,.2f} €")
print(f"{'=' * 60}")
print(f"  Importe retirado:     {df_tax.attrs.get('withdrawn_amount', 0):>12,.2f} €")
print(f"  Ganancia patrimonial: {df_tax.attrs.get('total_capital_gain', 0):>12,.2f} €")
print(f"  Impuestos estimados:  {df_tax.attrs.get('estimated_tax', 0):>12,.2f} €")
print(f"  Neto tras impuestos:  {df_tax.attrs.get('net_amount', 0):>12,.2f} €")
print(f"{'=' * 60}")

df_tax

Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\app\data\cache

--- Procesando: ES0141116030 (light)---
✗ Error obteniendo datos para ES0141116030: Invalid ISIN number: ES0141116030
  [Error] obteniendo NAV ligero para ES0141116030: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  RETIRADA OPTIMIZADA: 50,000.00 €
  Importe retirado:        50,000.00 €
  Ganancia patrimonial:     1,034.93 €
  Impuestos estimados:        196.64 €
  Neto tras impuestos:     49,803.36 €


,ISIN,Fondo,Fecha_Compra,Participaciones_Vendidas,Importe_Retirado,Ganancia_Patrimonial
0,LU0273159177,LU0273159177,2026-01-29 00:00:00,3.16,941.56,-58.44
1,LU0273159177,LU0273159177,2026-01-30 00:00:00,6.32,"1,883.13",-116.87
2,LU0273159177,LU0273159177,2026-01-31 00:00:00,6.06,"1,804.22",-195.78
3,LU0273159177,LU0273159177,2026-02-25 00:00:00,5.38,"1,602.03",-416.99
4,LU3038481936,LU3038481936,2026-02-24 00:00:00,6.94,"1,961.50",-38.50
5,LU3038481936,LU3038481936,2026-04-15 00:00:00,5.32,"1,502.35",2.35
6,LU0840158819,LU0840158819,2026-02-09 00:00:00,32.27,"5,065.73",65.73
7,LU1988110927,LU1988110927,2025-08-27 00:00:00,"1,533.27","2,028.21",28.21
8,LU1694789451,LU1694789451,2025-08-28 00:00:00,19.50,"2,539.74",39.74
9,IE00BH6XSF26,IE00BH6XSF26,2026-02-09 00:00:00,16.58,"5,176.09",176.09


In [ ]:
# Desglose fiscal por tramos
TAX_BRACKETS = [
    (6_000, 0.19, "0-6K €"),
    (50_000, 0.21, "6K-50K €"),
    (200_000, 0.23, "50K-200K €"),
    (300_000, 0.27, "200K-300K €"),
    (float("inf"), 0.28, ">300K €"),
]

capital_gain = df_tax.attrs.get("total_capital_gain", 0)

if capital_gain > 0:
    remaining = capital_gain
    prev_limit = 0
    bracket_data = []
    for limit, rate, label in TAX_BRACKETS:
        if remaining <= 0:
            break
        bracket_size = limit - prev_limit
        aplicable = min(remaining, bracket_size)
        tax = aplicable * rate
        bracket_data.append({"Tramo": label, "Tipo": f"{rate * 100:.0f}%", "Base": aplicable, "Cuota": tax})
        remaining -= aplicable
        prev_limit = limit

    df_brackets = pd.DataFrame(bracket_data)
    display(df_brackets)

    fig = go.Figure(go.Bar(
        x=[d["Tramo"] for d in bracket_data],
        y=[d["Cuota"] for d in bracket_data],
        text=[f"{d['Cuota']:,.2f} €" for d in bracket_data],
        textposition="outside",
        marker_color=["#ffd93d", "#ff8a5c", "#ff6b6b", "#e17055", "#d63031"][:len(bracket_data)],
    ))
    fig.update_layout(
        title="Desglose Fiscal por Tramos (Ahorro España 2024)",
        template="plotly_dark",
        yaxis_title="Impuesto (€)",
        height=400,
    )
    fig.show()
else:
    print("Sin ganancia patrimonial — no se generan impuestos.")

,Tramo,Tipo,Base,Cuota
0,0-6K €,19%,"1,034.93",196.64


## 10. Performance del Portfolio

In [ ]:
# Métricas de rendimiento del portfolio
df_perf = client.performance(years=5)
df_perf

Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0141116030 (light)


,Metric,Value


## 11. Diagnóstico de Datos

In [ ]:
# Diagnóstico de cobertura de datos por fondo
df_diag = client.diagnostics(years=5)
df_diag

Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0141116030 (light)


,ISIN,Fondo,Puntos_NAV,Desde,Hasta,Num_Sectores,Num_Regiones,Tiene_TER,Tiene_Sharpe,Estado
0,ES0141116030,ES0141116030,0,NaT,NaT,0,0,False,False,Sin datos
1,IE00BYX5MX67,Fidelity S&P 500 Index Fund,1,2025-06-11,2025-06-11,11,9,True,True,Poco historial
2,LU0273159177,DWS Invest - Gold and Precious Metals Equities,1,2026-05-04,2026-05-04,2,10,True,True,Poco historial
3,LU0302296495,DNB Fund - Technology,1,2026-05-04,2026-05-04,9,9,True,True,Poco historial
4,LU0329355670,Robeco QI Emerging Markets Active Equities,1,2026-05-04,2026-05-04,11,10,True,True,Poco historial
5,LU3038481936,Hamco SICAV Global Value,92,2025-12-01,2026-04-29,10,14,True,True,OK
6,IE000ZYRH0Q7,iShares Developed World Index Fund (IE),164,2025-08-26,2026-04-30,11,14,True,False,OK
7,LU2466448532,Echiquier Space,815,2022-12-15,2026-04-29,9,11,True,True,OK
8,LU1988110927,Buy & Hold Luxembourg B&H Bond,1007,2022-03-07,2026-04-29,0,10,True,True,OK
9,LU1598719752,Cobas LUX SICAV - Cobas International Fund,1011,2022-03-07,2026-04-29,10,13,True,True,OK
